# 11 Regression V2 — 적정가 예측 · 오차 범위 · 적정가 대비 등급 · OOF Gap

## 목적

| 결과 | 쓰임 |
|---|---|
| **적정가** (거래 단위 예측가) | 웹 단지 화면의 "모델 적정가" |
| **80% 오차 범위** | "적정가 범위" 표시, 등급 판정 기준 |
| **적정가 대비 등급** (규칙) | 저평가 ~ 고평가 5단계 (모델 기준 표현, 매수 · 매도 권유 없음) |
| **OOF Gap 구 × 월** | 31 분류 Feature 후보 (회귀 → 분류 연결) |

## 결론 요약

- **모델:** LightGBM 단일 모델 (Optuna 튜닝), 정답 `log(거래금액)`, 기본 Feature 199개 (`v2_feature_registry.csv` role=model).
- **평가는 서비스 방식(매달 재학습)을 기준으로 한다.** 매달 m월 시작 시점에 확정된 거래(m-2월까지)로 다시 학습해 m월을 예측한다.
- **기준선은 사용자가 이미 보는 값 — 같은 단지 · 같은 면적 직전 실거래가**다. 첫 버전은 이 기준선과 거의 같아서(중앙 오차 5.23 vs 5.26%),
  직전 거래 Feature 5개를 06에 추가하고 다시 튜닝했다 (5-2장).

| 지표 (매달 재학습, 2026-01~06 Test, 중개거래) | 값 |
|---|---:|
| MAPE / 중앙 오차 (전체 중개거래) | **5.53% / 4.22%** |
| 직전 거래가 있는 거래(97.7%): 모델 vs 같은 단지 · 면적 직전 거래가 | **5.37% vs 6.67%** (중앙 4.16 vs 4.84%) |
| 같은 기간 단순 규칙 (단지 최근 12개월 평당가 × 평) | 14.94% |
| 80% 구간 포함률 (기존 단지) | 80% 이상 (7장) |

- 고정 분할(2025-06까지 학습 → 2026 상반기)로는 보완 전 모델 Test MAPE 9.64%다. 6~12개월 재학습 없이 오른 시장을 따라가지 못한 값이다.
  같은 조건에서 V1 Feature는 13.26%, V2 Feature는 9.92%다.
- 앙상블 · 정답 변환 · 최근 오차 보정은 **채택 기준(두 fold 모두 개선, 홀드아웃 악화 없음)**을 넘지 못해 쓰지 않는다.
- 신규 단지(거래 이력 없음)는 구간을 넓혀도 80%를 맞추지 못해 **등급을 매기지 않고 범위만 참고용으로** 보여준다.

## 원칙

- 분할: Train 2021-07~2025-06, Valid 2025-07~12, Test 2026-01~06 (V1과 동일). 선택은 Train/Valid로만 하고 Test는 확인용이다.
- 실험(탐색 · 재학습 수 시간)은 `v2_experiments/` 스크립트로 돌렸고, 이 노트북은 그 결과를 불러와 해석한다.
  오차 범위 · 등급 · OOF Gap · 최종 모델은 여기서 직접 계산한다.
- Seed만 바꿔도 Valid MAPE가 6.34~6.78%로 흔들렸다 (학습률 0.1). **0.4%p보다 작은 차이는 우연일 수 있다**고 보고 판단한다.
- 서비스 기준 모집단은 **중개거래**다. 직거래는 가족 간 거래 등 시세와 다른 가격이 섞여 오차가 2배다 (따로 보고).

In [1]:
from pathlib import Path
import gzip
import json
import sqlite3

import lightgbm as lgb
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)


def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "data").exists() and (path / ".env").exists():
            return path
    raise FileNotFoundError("프로젝트 루트를 찾을 수 없습니다.")


PROJECT_DIR = find_project_root()
PROCESSED_DIR = PROJECT_DIR / "data" / "v2_5year_seoul" / "processed"
EXP_DIR = PROJECT_DIR / "notebook" / "v2_5year_notebook" / "model" / "regression" / "v2_experiments"
MODEL_DIR = PROJECT_DIR / "models" / "regression" / "v2"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

registry = pd.read_csv(PROCESSED_DIR / "v2_feature_registry.csv")
registry = registry[registry["dataset"].eq("v2_trade")]
FEATURES = registry.loc[registry["role"].eq("model"), "column"].tolist()
trade = pd.read_parquet(PROCESSED_DIR / "v2_trade.parquet")
trade["단지일련번호"] = trade["단지일련번호"].astype(str)
CATEGORICAL = [c for c in FEATURES if trade[c].dtype == object]

print("기본 Feature", len(FEATURES), "| 범주형", CATEGORICAL)
print(trade.groupby("데이터구분")["거래월일자"].agg(["min", "max", "count"]))

기본 Feature 199 | 범주형 ['거래유형', '토지임대부아파트여부']
             min        max   count
데이터구분                              
test  2026-01-01 2026-06-01   39569
train 2021-07-01 2025-06-01  160945
valid 2025-07-01 2025-12-01   33788


## 1. 기준선과 Feature 묶음 비교

같은 빠른 설정(XGBoost, 학습률 0.1, 깊이 6)으로 Feature 묶음을 하나씩 더했다 (`r11_stage1.py`).

- **Seen/Unseen:** Valid 거래 중 Train에 있던 단지 / Valid에서 처음 거래된 단지(950건).
- **단지 홀드아웃:** Train 단지 20%를 학습에서 빼고, 그 단지들의 Valid 거래로 평가 (3회 평균). 거래 이력은 있지만 모델이 처음 보는 단지다.
- **단순 규칙:** 같은 단지 확정 12개월(m-13~m-2) 중위 평당가 × 평, 없으면 동 → 구.

In [2]:
stage1 = pd.read_csv(EXP_DIR / "r11_stage1.csv")
naive = json.load(open(EXP_DIR / "r11_naive.json", encoding="utf-8"))
print("단순 규칙:", {k: round(v, 2) if isinstance(v, float) else v for k, v in naive.items()})
stage1.drop(columns=["초"]).round(2)

단순 규칙: {'구성': '단순 규칙 (단지 확정 12개월 중위 평당가 × 평)', 'Valid_MAPE': 13.45, 'Test_MAPE': 14.94, 'Seen_MAPE': 12.72, 'Unseen(Valid 신규)_MAPE': 38.73, '단지값 사용 비율(Valid)': 0.96}


,구성,Feature수,best_iter,Valid_MAPE,Valid_중앙APE,Valid_10%이내,Seen_MAPE,Unseen_MAPE(Valid 신규),홀드아웃_MAPE,홀드아웃_MAPE_범위
0,V1raw (V1 시점 그대로),56,1612,9.25,7.06,64.19,9.11,13.97,12.42,11.92~13.02
1,F0 V1정의+V2시점 (출처·응급·유치원·학교거리 포함),60,2338,8.64,6.40,67.95,8.43,15.59,12.22,11.79~12.59
2,F1 핵심 (레지스트리 기본),52,1778,9.14,6.80,65.79,8.89,17.58,12.59,11.84~13.36
3,F1+공시,57,1437,6.71,5.11,79.52,6.63,9.47,7.27,7.21~7.34
4,F2 +단지속성,68,1671,8.05,6.07,70.54,7.89,13.61,10.71,10.41~11.16
5,F3 +단지속성+공시,73,1515,6.57,5.05,80.48,6.50,8.97,7.12,6.98~7.24
6,F4 +학교·학원,102,1952,6.51,4.91,80.58,6.39,10.60,7.20,7.16~7.22
7,F5 +구×월 (전체 196),196,1155,6.48,4.84,80.85,6.25,14.31,6.91,6.79~7.08
8,F6 +군집 라벨,198,883,6.50,4.90,80.52,6.32,12.76,6.97,6.94~6.99


- **시점을 바로잡아도 성능이 떨어지지 않는다.** 같은 V1 Feature에서 V1 시점 9.25% → V2 시점(확정월) 8.64%.
- **개선의 대부분은 공시가격이다.** 핵심 9.14% → +공시 6.71%, 단지 홀드아웃 12.59% → 7.27%.
  `공시기반_추정가`는 공시 ㎡당 가격 × 전용면적이라 실거래가가 섞이지 않고, 계약일 기준 이미 공시된 연도만 쓴다
  (2025-06 거래 → 2024년 공시, 2025-07 거래 → 2025년 공시, 05b · 06에서 검증).
- **군집 라벨은 효과가 없다** (6.48 → 6.50). V1과 같은 결론이라 쓰지 않는다.
- **새 문제:** 묶음을 더할수록 Valid 신규 단지 오차가 커진다 (단지속성+공시 8.97% → +학교 10.60% → +구×월 14.31%) → 2장.

## 2. Ablation — 예상가격 대체값 문제

`r11_check.py`로 Valid 신규 단지를 보면 66%가 준공 1년 이내, 64%가 공시가격이 아직 없다.
이 단지들의 `예상가격_만원`은 단지 이력이 없어 **동 · 면적대 → 구 중위값으로 대체**된다. 신축은 주변 구축보다 비싸서 대체값이 기준가를 틀리게 준다.

In [3]:
print(open(EXP_DIR / "r11_check_output.txt", encoding="utf-8").read().split("묶음 permutation")[0])
ablation = pd.read_csv(EXP_DIR / "r11_ablation.csv").drop(columns=["초"])
abl2 = pd.read_csv(EXP_DIR / "r11_abl2.csv")
display(ablation.round(2))
abl2.round(2)

Valid 신규 단지 거래 950 | 단지 293 | 연식 중앙 0.0 | 신축(연식<=1) 비율 0.661 | 거래유형 {'직거래': 0.65, '중개거래': 0.35}
공시 결측 비율(신규) 0.642 | 전체 Valid 0.027
F3 +단지속성+공시 신규 단지 APE 중앙 6.61 평균 8.97 | 상위 5% 제외 평균 7.48
F5 +구×월 (전체 196) 신규 단지 APE 중앙 11.68 평균 14.31 | 상위 5% 제외 평균 12.87
         F3           F5       
      count   mean count   mean
공시없음                           
False   340  11.72   340  11.82
True    610   7.44   610  15.70
                       F3     F5  diff
단지            구                       
용산남영역롯데캐슬헤리티지 용산구     3.4   16.5  13.1
계명(350-34)    강서구    12.8   26.3  13.5
라봄성동          성동구     3.9   18.2  14.3
(1-10)        성동구   116.9  134.9  18.0
고명            성북구    18.1   36.3  18.3
LOFT129       동대문구    6.4   30.3  24.0
이수스위첸포레힐즈     동작구     3.7   28.0  24.3
글로리안(A)       중구     52.4   90.1  37.7



,구성,Feature수,Valid_MAPE,Valid_중앙APE,Seen_MAPE,Valid신규_MAPE,홀드아웃_MAPE
0,A0 전체(196),196,6.48,4.84,6.25,14.31,6.91
1,A1 예상가격 계열 제외,193,6.77,5.12,6.66,10.42,7.31
2,A2 + 예상가격_출처,197,6.87,5.25,6.71,12.16,6.95
3,A3 + 전세피처출처,199,6.50,4.83,6.28,14.07,6.93
4,A4 + 응급·유치원 거리,198,6.56,4.95,6.39,12.27,6.97
5,A5 공급 재고비율 → 원수치,196,6.79,5.18,6.61,13.22,6.93
6,A6 역 출입구 제외(역 좌표만),195,6.63,4.88,6.44,13.28,6.92
7,A7 역 좌표 제외(출입구만),195,6.53,4.84,6.31,14.01,6.82
8,A8 시장·전월세·금리 V1 시점(_v1),199,6.40,4.75,6.21,13.03,6.89
9,A9 15억 판정 제외,193,6.45,4.80,6.27,12.68,6.98


,구성,Feature수,Valid_MAPE_s42,Valid_MAPE_s43,Valid_MAPE_s44,Seen_MAPE,Valid신규_MAPE,신규_단지,신규_동·면적대,신규_구,홀드아웃_MAPE
0,A0 전체(196) 시드 3개,196,6.48,6.34,6.78,6.25,14.31,11.41,14.29,15.71,NaN
1,A1 예상가격 계열 제외,193,6.77,NaN,NaN,6.66,10.42,12.04,10.16,11.57,NaN
2,A12 예상가격 → 단지 이력만,196,6.37,NaN,NaN,6.26,10.00,9.77,10.18,8.82,6.77
3,A13 A12 + 주담대 계열 제외,194,6.28,NaN,NaN,6.19,9.35,10.12,8.94,11.93,6.68


| 비교 | 판단 |
|---|---|
| A0 시드 3개 | Valid 6.34~6.78% — **0.4%p 안쪽 차이는 우연 범위** |
| A1 예상가격 계열 제외 | 신규 14.3 → 10.4%로 좋아지지만 기존 단지 · 홀드아웃이 나빠짐 |
| A2 예상가격 출처 표시 추가 | 해결 안 됨 |
| A3~A9 (전세 출처 · 응급 · 유치원 · 공급 원수치 · 역 좌표/출입구 · V1 시점 · 15억) | 우연 범위 → 기존 기본값 유지 (시점이 정직한 쪽) |
| A10 · A11 permutation 0 이하 59개 제외 | 나빠짐 → 일괄 제외 안 함 |
| **A13 예상가격은 단지 자체 이력일 때만 + 대체값 기반 주담대 한도 2개 제외** | **Valid 6.28 · 신규 9.35 · 홀드아웃 6.68 — 모든 지표 개선 → 채택** |

06에 `예상가격_단지만`을 추가하고 `예상가격_만원` · `주담대_최대한도_만원` · `주담대_한도비율`을 `ablation`으로 옮겼다 (기본 Feature 194개, 5-2장 보완 후 199개).

In [4]:
LAST_TRADE = registry.loc[registry["source"].eq("06_last_trade"), "column"].tolist()
assert set(FEATURES) == set(json.load(open(EXP_DIR / "r11_features.json", encoding="utf-8"))) | set(LAST_TRADE), "기본 Feature ≠ A13 + 직전 거래"
assert "예상가격_단지만" in FEATURES and "예상가격_만원" not in FEATURES
own = trade["예상가격_출처"].eq("단지")
assert trade.loc[own, "예상가격_단지만"].equals(trade.loc[own, "예상가격_만원"]) and trade.loc[~own, "예상가격_단지만"].isna().all()
print(f"✅ 기본 Feature = A13 194개 + 직전 거래 {len(LAST_TRADE)}개 (5-2장), 예상가격_단지만 = 단지 이력일 때만 예상가격")

✅ 기본 Feature = A13 194개 + 직전 거래 5개 (5-2장), 예상가격_단지만 = 단지 이력일 때만 예상가격


## 3. 하이퍼파라미터 탐색 (보완 전 194개)

Optuna TPE, 학습률 0.1 고정, **시간 fold 2개 평균 MAPE**를 최소화한다 (`r11_tune.py`).

- fold 1: 2024-12까지 학습 → 2025H1 평가 / fold 2: 2025-06까지 학습 → 2025H2(Valid) 평가
- fold 1이 중위값보다 나쁘면 중간 종료 (MedianPruner). XGBoost는 1단계 기본 설정을 첫 시도로 넣었다.

In [5]:
con = sqlite3.connect(EXP_DIR / "r11_optuna.db")
trials = pd.read_sql(
    "select s.study_name, t.number, t.state, v.value from trials t join studies s using(study_id) "
    "left join trial_values v using(trial_id)", con)
attrs = pd.read_sql("select t.number, s.study_name, a.key, a.value_json from trial_user_attributes a "
                    "join trials t using(trial_id) join studies s using(study_id)", con)
con.close()
summary = trials.groupby("study_name").agg(시도=("number", "count"), 완료=("state", lambda s: (s == "COMPLETE").sum()),
                                           최적=("value", "min"))
best_rows = trials.loc[trials.groupby("study_name")["value"].idxmin()]
folds = attrs[attrs["key"].eq("fold_mape")].merge(best_rows, on=["study_name", "number"])
summary["최적 fold1 · fold2"] = folds.set_index("study_name")["value_json"].map(lambda v: [round(x, 2) for x in json.loads(v)])
first_xgb = trials[(trials.study_name == "r11_xgb") & (trials.number == 0)]["value"].iloc[0]
print("XGBoost 1단계 기본 설정(첫 시도):", round(first_xgb, 3))
best_params = json.load(open(EXP_DIR / "r11_best_params.json"))
print(json.dumps(best_params, indent=1, ensure_ascii=False))
summary.round(3)

XGBoost 1단계 기본 설정(첫 시도): 6.59
{
 "xgb": {
  "max_depth": 12,
  "min_child_weight": 31.742177483769634,
  "subsample": 0.6662357438035773,
  "colsample_bytree": 0.9641507700604032,
  "reg_lambda": 0.08221832913402648,
  "reg_alpha": 0.19721690639475783,
  "max_cat_to_onehot": 15
 },
 "lgb": {
  "num_leaves": 451,
  "min_child_samples": 12,
  "colsample_bytree": 0.7268667382517547,
  "subsample": 0.9648682390881465,
  "reg_lambda": 0.03963700922832873,
  "reg_alpha": 0.45659408478817315,
  "cat_smooth": 65.52911519057434,
  "max_bin": 511
 },
 "cat": {
  "depth": 8,
  "l2_leaf_reg": 32.74127435518369,
  "random_strength": 4.33744643345235,
  "bagging_temperature": 0.3834415188257777,
  "rsm": 0.8542075266578653,
  "one_hot_max_size": 8
 }
}


,시도,완료,최적,최적 fold1 · fold2
study_name,,,,
r11_cat,20,13,6.565,"[6.45, 6.68]"
r11_lgb,30,13,6.338,"[6.33, 6.35]"
r11_lgb_lasttrade,30,14,5.476,"[5.58, 5.37]"
r11_xgb,30,14,6.433,"[6.44, 6.42]"


- LightGBM이 가장 좋다 (6.34%). XGBoost는 기본 설정 대비 0.16%p 개선으로 흔들림 폭 안이다.
- CatBoost는 한 번 학습에 최대 50분이 걸리고 성능도 가장 낮다.

## 4. 최종 학습률 재학습과 앙상블 (보완 전 194개)

최적 설정을 최종 학습률(XGB · LGB 0.03, CatBoost 0.1)로 fold 2개 + 단지 홀드아웃 3개에 다시 학습했다 (`r11_refit.py`).
앙상블 후보: LightGBM 시드 3개 배깅, 정답 다양화(`log(평당가)` LightGBM), 균등 · 가중 블렌딩 (`r11_blend.py`).
가중 블렌딩 가중치는 **다른 fold에서 학습해 교차 적용**하고, 홀드아웃은 Valid를 보지 않은 fold 1 가중치로 평가했다.

**채택 기준:** 최고 단일 모델보다 두 fold 모두 0.1%p 이상 좋아지고 홀드아웃이 나빠지지 않을 때만.

In [6]:
refit = pd.read_json(EXP_DIR / "r11_refit_log.jsonl", lines=True)
display(refit.pivot(index="job", columns="split", values="mape").round(3))
blend = pd.read_csv(EXP_DIR / "r11_blend.csv")
blend.head(15)

split,f1,f2,h1,h2,h3
job,,,,,
cat_s42,7.382,7.088,7.082,7.518,7.147
lgb_py_s42,6.880,6.455,7.155,6.972,6.991
lgb_s42,6.486,6.160,6.598,6.606,6.667
lgb_s43,6.521,6.256,6.653,6.570,6.607
lgb_s44,6.588,6.189,6.650,6.623,6.631
xgb_s42,6.550,6.314,6.616,6.640,6.637


,구성,fold1,fold2(Valid),fold 평균,Valid 기존 단지,Valid 신규 단지,Valid 10% 이내,홀드아웃,비고
0,lgb_s42,6.486,6.160,6.323,6.003,11.584,82.236,6.624,NaN
1,가중 lgb_bag3+xgb_s42,6.494,6.170,6.332,6.003,11.928,82.213,6.546,"w(f1)=[0.67, 0.33] w(f2)=[0.88, 0.12]"
2,가중 lgb_bag3+xgb_s42+cat_s42,6.494,6.169,6.332,6.003,11.927,82.213,6.546,"w(f1)=[0.67, 0.33, 0.0] w(f2)=[0.88, 0.12, 0.0]"
3,가중 lgb_bag3+cat_s42,6.503,6.163,6.333,6.004,11.677,82.139,6.583,"w(f1)=[1.0, 0.0] w(f2)=[1.0, 0.0]"
4,lgb_bag3,6.503,6.163,6.333,6.004,11.677,82.139,6.583,NaN
5,균등 lgb_bag3+xgb_s42,6.491,6.189,6.340,6.019,12.064,82.112,6.548,NaN
6,가중 lgb_bag3+xgb_s42+lgb_py_s42,6.530,6.170,6.350,6.003,11.928,82.213,6.546,"w(f1)=[0.67, 0.33, 0.0] w(f2)=[0.7, 0.11, 0.19]"
7,가중 lgb_bag3+lgb_py_s42,6.538,6.163,6.350,6.004,11.677,82.139,6.583,"w(f1)=[1.0, 0.0] w(f2)=[0.81, 0.19]"
8,가중 lgb_bag3+cat_s42+lgb_py_s42,6.538,6.163,6.350,6.004,11.677,82.139,6.583,"w(f1)=[1.0, 0.0, 0.0] w(f2)=[0.81, 0.0, 0.19]"
9,가중 lgb_bag3+xgb_s42+cat_s42+lgb_py_s42,6.530,6.169,6.350,6.003,11.928,82.213,6.546,"w(f1)=[0.67, 0.33, 0.0, 0.0] w(f2)=[0.7, 0.11,..."


- 단일 LightGBM(시드 42)보다 두 fold 모두 0.1%p 이상 나은 조합이 없다. 시드 배깅은 단일 시드 기대값보다 0.03~0.04%p 낫지만 기준 미달.
- 4개 균등 블렌드는 Valid 신규 단지를 11.7 → 9.9%로 줄이지만 전체는 나빠진다.
  신규 단지만 균등 블렌드로 보내는 분기(`r11_route.py`)는 fold 2에서만 좋아지고 fold 1에서 나빠져 (11.28 → 11.47) 쓰지 않는다.
- CatBoost는 같은 설정으로 다시 학습해도 탐색 때보다 크게 나빴다 (fold 평균 6.56 → 7.24). 재현이 불안정해 후보에서 뺀다.
- **결정: LightGBM 단일 모델.** (설정은 5-2장 보완 후 다시 탐색)

## 5. 평가 방식 — 고정 분할 vs 매달 재학습

고정 분할 Test에서 MAPE가 Valid보다 크게 나빠졌다 (6.16 → 9.64%). 모든 달에서 예측이 실제보다 평균 6% 낮다.

| 진단 | 결과 |
|---|---|
| 같은 단지 실제가 / 30~395일 전 중위가 | Valid 1.056 → Test 1.074 — 2026년 상승 속도가 더 빠름 |
| 과소추정 큰 구 | 성동 -14.6% · 동작 -13.7% · 송파 -10.1% · 광진 -10.0% (2026 상승 지역) |

시장 수준 대비 정답(구 시세 · 단지 최근가 대비 비율)과 Train+Valid 재학습을 비교했다 (`r11_drift.py`).
같은 고정 분할 · 같은 빠른 설정으로 V1 Feature와도 비교했다 (`r11_v1cmp.py`, V1 노트북의 9.37%는 기준월 당월 정보가 섞인 값이라 직접 비교하지 않는다).

In [7]:
display(pd.read_csv(EXP_DIR / "r11_drift.csv").round(2))
pd.read_csv(EXP_DIR / "r11_v1cmp.csv")

,구성,Valid_MAPE,Valid_편향,Test_MAPE,Test_편향
0,abs | Train 학습,6.16,-0.60,9.64,-6.02
1,abs | Train+Valid 재학습 (3132회),NaN,NaN,8.05,-4.65
2,구평당가 | Train 학습,7.34,-2.35,11.43,-7.88
3,구평당가 | Train+Valid 재학습 (2386회),NaN,NaN,9.26,-6.14
4,단지예상가 | Train 학습,6.20,-1.12,9.41,-5.04
5,단지예상가 | Train+Valid 재학습 (3119회),NaN,NaN,8.16,-3.91


,구성,Feature수,Valid_MAPE,Test_MAPE,Test_편향
0,V1 시점 그대로,56,9.25,13.26,-8.78
1,V1 정의 + V2 시점,60,8.64,12.46,-7.72
2,V2 기본(A13),194,6.48,9.92,-5.65


- 정답 변환은 편향을 조금 줄이지만 MAPE가 나아지지 않는다 → **정답은 `log(거래금액)` 유지.**
- 같은 조건에서 **V2가 V1보다 Test 3.3%p 낫다** (13.26 → 9.92%, 학습률 0.1 기준).
- 서비스는 매달 재학습하므로, 서비스 성능은 **매달 재학습 평가**로 잰다:
  m월 예측 = m월 1일에 확정된 거래(≤ m-2월)로 학습한 모델. 2022-01~2026-06 54개월 모두 이 방식으로 예측했다 (OOF).
  보완 전 모델(`r11_rolling.py`): Valid 6.06% · Test 6.77% (중개 5.70 · 6.52%).

### 최근 확정월 보정 — 기각 (보완 전 모델로 확인)

m-2월 OOF 잔차(서울 또는 구 중위)를 m월 예측에 더하는 보정은 Test에서 6.77 → 6.36%로 좋아졌지만,
**Valid에서는 6.06 → 6.59%로 나빠졌다.** Test를 보고 고르는 셈이 되므로 쓰지 않는다.

In [8]:
rolling = pd.read_csv(EXP_DIR / "r11_rolling.csv")
rolling["구간"] = np.select([rolling["월"].between("2025-07", "2025-12"), rolling["월"] >= "2026-01"], ["Valid", "Test"], "")
rolling = rolling[rolling["구간"] != ""]
rolling.groupby("구간")[["n", "MAPE", "MAPE_서울보정", "MAPE_구보정"]].apply(
    lambda g: pd.Series({c: np.average(g[c], weights=g["n"]) for c in ["MAPE", "MAPE_서울보정", "MAPE_구보정"]})).round(3)

,MAPE,MAPE_서울보정,MAPE_구보정
구간,,,
Test,6.773,6.355,6.412
Valid,6.058,6.585,6.506


## 5-2. 보완 — 같은 단지 · 같은 면적 직전 거래

보완 전 모델을 **사용자가 이미 보는 값**(같은 단지 · 같은 평형 최근 실거래가)과 비교하니 거의 같았다
(Test 중개거래, 직전 거래 있는 거래: MAPE 6.36 vs 6.92%, 중앙 5.23 vs 5.26%). 모델은 단지 전체 1년 중위값(`예상가격_단지만`)만 알고
같은 면적의 가장 최근 거래라는 가장 강한 신호를 몰랐다.

- 06에 Feature 5개 추가 (`06_last_trade`): 직전 거래가 · 직전 3건 중위 · 경과일 · 직전 거래 층 · 같은 면적 누적 거래 수.
  계약일 30일 전까지 계약된 중개거래만 (신고 기한 30일, `예상가격`과 같은 규칙), 경계일 골든 테스트 포함.
- 빠른 확인 (Train → Valid, 같은 설정): 아래 출력. 시드 흔들림(0.4%p)의 2배가 넘는 개선이라 채택.
- LightGBM 재탐색 30회 (`r11_tune2.py`, 보완 전 최적 설정을 첫 시도로): fold 평균 5.52 → 5.48%로 차이는 작지만,
  잎 수가 451 → 72로 줄어 모델이 작고 빠르다 → 새 설정 채택. 반복 수는 Train → Valid 조기 종료 × 1.15.
- 같은 방식으로 54개월 매달 재학습을 다시 했다 (`r11_rolling2.py`).

In [9]:
print(open(EXP_DIR / "lt_test_output.txt", encoding="utf-8").read().replace("  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)\n", ""))
con = sqlite3.connect(EXP_DIR / "r11_optuna.db")
lt = pd.read_sql("select t.number, t.state, v.value from trials t join studies s using(study_id) left join trial_values v using(trial_id) "
                 "where s.study_name = 'r11_lgb_lasttrade'", con)
con.close()
final_params = json.load(open(EXP_DIR / "r11_best_params_lasttrade.json", encoding="utf-8"))
print("재탐색:", len(lt), "회 | 첫 시도(보완 전 설정)", round(lt.loc[lt.number == 0, "value"].iloc[0], 3), "| 최적", round(lt["value"].min(), 3))
print(json.dumps(final_params, indent=1, ensure_ascii=False))
compare = pd.read_csv(EXP_DIR / "r11_lasttrade_compare.csv")
compare

기존 194 iter 2724 Valid 6.16 중개 5.799 중개 중앙 4.365 신규 11.58 769 초
+ 직전 거래 5개 iter 857 Valid 5.242 중개 4.828 중개 중앙 3.491 신규 11.8 1489 초

[exited with code 0]

재탐색: 30 회 | 첫 시도(보완 전 설정) 5.521 | 최적 5.476
{
 "lgb": {
  "num_leaves": 72,
  "min_child_samples": 24,
  "colsample_bytree": 0.4322311163924917,
  "subsample": 0.9944803483540686,
  "reg_lambda": 46.27272675640742,
  "reg_alpha": 0.010095396757182767,
  "cat_smooth": 16.406899587916314,
  "max_bin": 255
 },
 "n_estimators": 4290,
 "learning_rate": 0.03,
 "train_valid_mape": 5.329470182178547,
 "best_iteration": 3734
}


,방법,구간,직전 거래 있는 거래 MAPE,직전 거래 있는 거래 중앙 APE,직전 거래 있는 거래 10%이내,전체 MAPE,직전 거래 없는 거래 MAPE,직전 거래 있음 비율
0,보완 후 모델,Valid,4.840,3.594,89.484,4.947,10.733,98.186
1,보완 후 모델,Test,5.374,4.157,87.433,5.525,12.091,97.738
2,보완 전 모델,Valid,5.609,4.291,84.612,5.696,10.383,98.186
3,보완 전 모델,Test,6.408,5.255,80.108,6.523,11.483,97.738
4,같은 단지·면적 직전 거래가,Valid,6.312,4.564,80.923,NaN,NaN,98.186
5,같은 단지·면적 직전 거래가,Test,6.665,4.839,79.317,NaN,NaN,97.738
6,같은 단지·면적 직전 3건 중위,Valid,6.490,4.762,79.911,NaN,NaN,98.186
7,같은 단지·면적 직전 3건 중위,Test,6.926,5.114,77.726,NaN,NaN,97.738


- **보완 후 모델이 직전 거래가를 이긴다:** Test 직전 거래 있는 거래 MAPE 5.37 vs 6.67%, 중앙 4.16 vs 4.84%, 10% 이내 87.4 vs 79.3%.
- 보완 전 모델보다 Test 1.0%p 낫다 (6.41 → 5.37%). 직전 거래가 없는 거래(2.3%)는 그대로 10~12%다.

In [10]:
oof = pd.read_parquet(MODEL_DIR / "oof_predictions.parquet")
oof["월"] = oof["월"].astype("period[M]")
oof["bias"] = (np.exp(oof["p"] - oof["y"]) - 1) * 100
VALID_M = pd.period_range("2025-07", "2025-12", freq="M")
TEST_M = pd.period_range("2026-01", "2026-06", freq="M")
oof["구간"] = np.select([oof["월"].isin(VALID_M), oof["월"].isin(TEST_M)], ["Valid", "Test"], "그 이전")
MARKET = oof["거래유형"].eq("중개거래")

perf = oof[oof["구간"] != "그 이전"].groupby(["구간", "거래유형"]).agg(n=("ape", "size"), MAPE=("ape", "mean"),
                                                                     중앙APE=("ape", "median"), 편향=("bias", "mean"))
perf["10%이내"] = oof[oof["구간"] != "그 이전"].groupby(["구간", "거래유형"])["ape"].apply(lambda s: (s <= 10).mean() * 100)
overall = oof[oof["구간"] != "그 이전"].groupby("구간").agg(n=("ape", "size"), MAPE=("ape", "mean"), 편향=("bias", "mean"))
display(overall.round(2))
display(perf.round(2))
print("연도별 중개거래 MAPE:", oof[MARKET].groupby(oof["월"].dt.year)["ape"].mean().round(2).to_dict())
oof[oof["구간"] != "그 이전"].groupby("월").agg(MAPE=("ape", "mean"), 편향=("bias", "mean")).round(2).T

,n,MAPE,편향
구간,,,
Test,39569,5.82,-1.88
Valid,33788,5.33,-1.20


n   MAPE  중앙APE    편향  10%이내
구간    거래유형                                  
Test  중개거래  37923   5.53   4.22 -2.10  86.64
      직거래    1646  12.51   9.25  3.15  52.49
Valid 중개거래  32193   4.95   3.64 -1.48  88.93
      직거래    1595  12.95   9.77  4.44  50.72

연도별 중개거래 MAPE: {2022: 8.8, 2023: 6.08, 2024: 4.5, 2025: 4.87, 2026: 5.53}


월,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06
MAPE,5.45,5.16,4.71,5.92,5.70,5.16,5.01,6.16,5.99,6.00,5.97,5.51
편향,-0.22,1.31,-1.39,-2.81,-2.18,-0.41,-1.65,-3.18,-2.36,-1.37,-1.00,-2.53


## 6. 오류가 큰 곳 (Test, 중개거래)

In [11]:
test_oof = oof[oof["구간"].eq("Test") & MARKET].merge(trade[["transaction_id", "시군구명", "연식"]], on="transaction_id")
display(test_oof.groupby(["단지", "가격대"]).agg(n=("ape", "size"), MAPE=("ape", "mean"), 편향=("bias", "mean")).round(2))
display(test_oof.groupby("시군구명").agg(n=("ape", "size"), MAPE=("ape", "mean"), 편향=("bias", "mean")).round(2)
        .sort_values("편향").head(6))
test_oof.groupby(pd.cut(test_oof["연식"], [-1, 1, 5, 15, 30, 100]), observed=True).agg(n=("ape", "size"), MAPE=("ape", "mean"), 편향=("bias", "mean")).round(2)

n   MAPE    편향
단지 가격대                       
기존 15억 이상   8212   5.03 -0.63
   6~9억    10445   5.30 -2.60
   6억 미만    7878   5.58 -1.43
   9~15억   11056   5.82 -3.31
신규 15억 이상     43  13.52 -1.16
   6~9억      103  13.61  2.97
   6억 미만     122  12.27  0.20
   9~15억      64  16.77  2.13

,n,MAPE,편향
시군구명,,,
동대문구,1639,5.84,-3.40
동작구,1308,5.80,-3.38
성동구,901,5.64,-3.19
강북구,896,4.93,-3.08
광진구,507,6.65,-2.93
성북구,2344,5.38,-2.92


,n,MAPE,편향
연식,,,
"(-1, 1]",186,8.28,-1.21
"(1, 5]",1964,5.24,-1.31
"(5, 15]",6028,4.86,-1.35
"(15, 30]",19346,5.56,-2.18
"(30, 100]",10399,5.84,-2.56


- 매달 재학습해도 과소추정이 남는다 (Test 중개 평균 -2.1%, 동대문 · 동작 -3.4% · 성동 -3.2%). 확정 거래가 2개월 늦게 들어오는 구조적 한계다.
  보완 전 모델(-3.2%, 성동 -6.2%)보다 줄었다 — 직전 거래가가 30일 전까지의 거래를 반영하기 때문이다.
  가격대로는 9~15억, 연식으로는 15년 이상 구축에서 편향이 크다.
- 신규 단지(학습 시점에 확정 거래가 없는 단지)는 오차가 크다 → 7장에서 따로 처리.

## 7. 80% 오차 범위

잔차 `r = log(실제) - log(예측)`의 분위수로 구간을 만든다. 구간은 OOF 잔차로만 만든다 (학습 거래 잔차는 오차를 과소평가).

- **세그먼트:** 단지(기존/신규) × 예측가 가격대(6억 미만 · 6~9억 · 9~15억 · 15억 이상)
- **이동 창:** m월 구간 = m-(창+1) ~ m-2월 중개거래 잔차 (m월 1일에 알 수 있는 값)
- **선택:** 창 길이(3/6/12개월) × 분위수 수준(0.10~0.05)을 **Valid에서** 골라 80% 이상 중 폭이 가장 좁은 것 → Test는 확인만

In [12]:
SEG = ["단지", "가격대"]


def interval_table(month, alpha, window, data=oof):
    w = data[data["거래유형"].eq("중개거래") & data["월"].between(month - (window + 1), month - 2)]
    q = w.groupby(SEG)["r"].quantile([alpha, 1 - alpha]).unstack()
    q.columns = ["lo", "hi"]
    q_all = w["r"].quantile([alpha, 1 - alpha]).to_numpy()
    return q, q_all


def attach_interval(rows, month, alpha, window):
    q, q_all = interval_table(month, alpha, window)
    key = pd.MultiIndex.from_frame(rows[SEG])
    out = rows.copy()
    out["lo"] = q["lo"].reindex(key).fillna(q_all[0]).to_numpy()
    out["hi"] = q["hi"].reindex(key).fillna(q_all[1]).to_numpy()
    out["포함"] = out["r"].between(out["lo"], out["hi"])
    out["폭%"] = (np.exp(out["hi"]) - np.exp(out["lo"])) * 100
    return out


def evaluate(months, alpha, window):
    return pd.concat([attach_interval(oof[oof["월"].eq(m)], m, alpha, window) for m in months], ignore_index=True)


grid = []
for window in (3, 6, 12):
    for alpha in (0.10, 0.08, 0.07, 0.06, 0.05):
        ev = evaluate(VALID_M, alpha, window)
        mk = ev["거래유형"].eq("중개거래") & ev["단지"].eq("기존")
        grid.append({"창": window, "분위수": alpha, "Valid 포함률(기존)": ev.loc[mk, "포함"].mean() * 100,
                     "Valid 폭%(기존)": ev.loc[mk, "폭%"].mean()})
grid = pd.DataFrame(grid)
chosen = grid[grid["Valid 포함률(기존)"] >= 80].sort_values("Valid 폭%(기존)").iloc[0]
WINDOW, ALPHA = int(chosen["창"]), float(chosen["분위수"])
print(f"선택: 창 {WINDOW}개월, 분위수 {ALPHA:.2f}~{1 - ALPHA:.2f}")
grid.round(2)

선택: 창 3개월, 분위수 0.08~0.92


,창,분위수,Valid 포함률(기존),Valid 폭%(기존)
0,3,0.10,76.43,13.38
1,3,0.08,80.82,14.95
2,3,0.07,83.11,15.93
3,3,0.06,85.50,17.08
4,3,0.05,87.81,18.46
5,6,0.10,75.11,13.22
6,6,0.08,79.61,14.75
7,6,0.07,81.92,15.69
8,6,0.06,84.30,16.79
9,6,0.05,86.65,18.08


In [13]:
test_iv = evaluate(TEST_M, ALPHA, WINDOW)
mk = test_iv["거래유형"].eq("중개거래")
print("Test 포함률 (중개, 기존 단지):", round(test_iv.loc[mk & test_iv["단지"].eq("기존"), "포함"].mean() * 100, 1),
      "| 평균 폭:", round(test_iv.loc[mk & test_iv["단지"].eq("기존"), "폭%"].mean(), 1))
display(test_iv[mk].groupby(SEG).agg(n=("포함", "size"), 포함률=("포함", "mean"), 폭=("폭%", "mean"))
        .assign(포함률=lambda d: d["포함률"] * 100).round(1))
test_iv[mk & test_iv["단지"].eq("기존")].groupby(test_iv["월"].astype(str))["포함"].mean().mul(100).round(1).to_frame("월별 포함률(기존)").T

Test 포함률 (중개, 기존 단지): 81.9 | 평균 폭: 16.6


n   포함률     폭
단지 가격대                      
기존 15억 이상   8212  83.0  17.3
   6~9억    10445  79.9  15.3
   6억 미만    7878  81.0  17.2
   9~15억   11056  83.6  17.1
신규 15억 이상     43  69.8  50.7
   6~9억      103  71.8  44.8
   6억 미만     122  82.8  39.7
   9~15억      64  68.8  45.3

월,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06
월별 포함률(기존),84.3,79.9,77.7,80.3,82.8,87.2


- **기존 단지:** Test 포함률 81.9% (가격대별 79.9~83.6%), 평균 폭 16.6% (보완 전 20.4%보다 좁다).
- **신규 단지:** 포함률 69~83%, 폭 40~51%. 보완 전 모델로 가격대를 합쳐 분위수를 Valid에서 따로 맞춰도(`r11_interval3.py`: Valid 82.5%) Test는 72%였다.
  거래 이력이 없는 단지는 구간을 넓혀도 믿을 만한 범위가 되지 않는다.
  → **신규 단지는 등급을 매기지 않고, 범위를 "거래 이력 없음 · 참고용"으로만 보여준다.**

## 8. 적정가 대비 등급 (규칙)

`r = log(실제 또는 호가 / 적정가)`를 구간 `[lo, hi]`와 비교한다. 초과 정도는 해당 방향 구간 반폭 기준이다.

| 등급 | 판정 |
|---|---|
| 적정 | `lo ≤ r ≤ hi` |
| 다소 고평가 / 다소 저평가 | 구간 밖, 초과분이 반폭의 1배 이하 (= 적정가에서 반폭의 1~2배) |
| 고평가 / 저평가 | 초과분이 반폭의 1배 초과 (= 반폭의 2배 초과) |
| 등급 없음 (참고용) | 신규 단지 |

실제 거래가가 범위보다 높으면 "고평가", 낮으면 "저평가"다. 사용자 표현은 "모델 기준"으로 고정한다.

In [14]:
def grade(r, lo, hi, complex_kind):
    if complex_kind == "신규":
        return "등급 없음(참고용)"
    if lo <= r <= hi:
        return "적정"
    if r > hi:
        return "다소 고평가" if (r - hi) <= hi else "고평가"
    return "다소 저평가" if (lo - r) <= -lo else "저평가"


test_iv["등급"] = [grade(a, b, c, k) for a, b, c, k in zip(test_iv["r"], test_iv["lo"], test_iv["hi"], test_iv["단지"])]
ORDER = ["저평가", "다소 저평가", "적정", "다소 고평가", "고평가", "등급 없음(참고용)"]
dist = pd.crosstab(test_iv["거래유형"], test_iv["등급"], normalize="index").mul(100).reindex(columns=ORDER).round(1)
dist

등급,저평가,다소 저평가,적정,다소 고평가,고평가,등급 없음(참고용)
거래유형,,,,,,
중개거래,3.0,5.7,81.2,8.4,0.8,0.9
직거래,18.0,5.8,28.3,11.8,3.9,32.3


- 중개거래는 81.2%가 적정이고, 구간 밖은 고평가 쪽 9.2% · 저평가 쪽 8.7%로 비슷하다.
- 직거래는 저평가 비율이 크다 — 가족 간 거래 등 시세보다 싼 거래가 섞인다는 해석과 맞는다. 웹에서 거래 목록에 등급을 붙일 때 직거래는 따로 표시한다.

## 9. OOF Gap 구 × 월 (31 분류 입력)

`price_gap_pct = 실제 / 예측 - 1`을 거래월 × 구로 집계한다. 모든 예측이 OOF(그 달 시작 시점에 확정된 거래로 학습)라 회귀 과적합이 섞이지 않는다.

- 거래월 M의 Gap은 신고 기한 때문에 **M+2월 1일부터** 알 수 있다 → `사용가능월` 열로 남기고, 31은 예측시점 p에 `사용가능월 ≤ p`인 값만 쓴다.
- 2022년은 학습 데이터가 짧아(첫 달 5개월치) OOF 오차가 크다 (중개 MAPE 9.4%). 31에서 Gap Feature는 Ablation으로 채택 여부를 정한다.

In [15]:
gap = oof.merge(trade[["transaction_id", "시군구명"]], on="transaction_id")
gap["gap"] = np.exp(gap["r"]) - 1
gu_month = (gap.groupby(["시군구명", "월"])
            .agg(거래수=("gap", "size"), gap_평균=("gap", "mean"), gap_중위=("gap", "median"))
            .join(gap[gap["거래유형"].eq("중개거래")].groupby(["시군구명", "월"])
                  .agg(중개_거래수=("gap", "size"), 중개_gap_중위=("gap", "median")))
            .reset_index())
gu_month["사용가능월"] = gu_month["월"] + 2
gu_month["월"] = gu_month["월"].dt.to_timestamp(); gu_month["사용가능월"] = gu_month["사용가능월"].dt.to_timestamp()
assert gu_month[["시군구명", "월"]].duplicated().sum() == 0 and gu_month["시군구명"].nunique() == 25
gu_month.to_parquet(MODEL_DIR / "oof_gap_gu_month.parquet", index=False)
print(gu_month.shape, gu_month["월"].min().date(), "~", gu_month["월"].max().date())
gu_month.pivot(index="월", columns="시군구명", values="중개_gap_중위").iloc[-6:, :8].round(3)

(1350, 8) 2022-01-01 ~ 2026-06-01


시군구명,강남구,강동구,강북구,강서구,관악구,광진구,구로구,금천구
월,,,,,,,,
2026-01-01,0.000,0.024,0.009,0.017,0.027,0.028,0.017,0.007
2026-02-01,0.024,0.040,0.034,0.046,0.051,0.049,0.026,0.009
2026-03-01,-0.012,0.030,0.037,0.043,0.031,0.014,0.027,0.005
2026-04-01,-0.031,0.019,0.051,0.026,0.025,0.018,0.029,0.012
2026-05-01,-0.006,0.019,0.036,0.028,0.022,0.037,0.020,0.011
2026-06-01,0.011,0.036,0.032,0.038,0.029,0.047,0.029,0.031


## 10. 최종 서비스 모델

2026-06까지 모든 거래로 학습한다 (2026-08 서비스 시점 기준 확정 거래). 설정 · 반복 수는 5-2장 매달 재학습 평가와 같다.
모델 파일은 gzip으로 저장한다 (GitHub 100MB 제한).

서비스용 오차 범위 표는 최근 3개 확정월(2026-04~06) OOF 잔차로 만든다 — 2026-08 예측에 쓰는 표다.

In [16]:
LGB_PARAMS = dict(final_params["lgb"], learning_rate=final_params["learning_rate"], n_estimators=final_params["n_estimators"],
                  bagging_freq=1, random_state=42, n_jobs=8, verbose=-1)
X = trade[FEATURES].copy()
for c in CATEGORICAL:
    X[c] = X[c].astype("category")
y = np.log(trade["거래금액"].to_numpy(float))
final = lgb.LGBMRegressor(**LGB_PARAMS).fit(X, y)
with gzip.open(MODEL_DIR / "lgbm_regression_v2.txt.gz", "wt", encoding="utf-8") as f:
    f.write(final.booster_.model_to_string())

# 저장한 파일로 다시 읽어 같은 예측이 나오는지 확인
with gzip.open(MODEL_DIR / "lgbm_regression_v2.txt.gz", "rt", encoding="utf-8") as f:
    reloaded = lgb.Booster(model_str=f.read())
sample = X.sample(2000, random_state=0)
assert np.allclose(reloaded.predict(sample), final.predict(sample))
print("✅ 저장 · 재로딩 예측 일치 |", round((MODEL_DIR / "lgbm_regression_v2.txt.gz").stat().st_size / 1e6, 1), "MB")

importance = pd.Series(final.booster_.feature_importance("gain"), index=FEATURES)
(importance / importance.sum() * 100).sort_values(ascending=False).head(20).round(2).to_frame("gain 비중 %")

✅ 저장 · 재로딩 예측 일치 | 11.0 MB


,gain 비중 %
직전거래가_동일면적,32.87
직전3건중위_동일면적,25.62
공시기반_추정가,17.80
전용면적,8.71
예상가격_단지만,5.92
공시_m2당_단지,3.58
전용_중위,0.76
공시_m2당_면적대,0.45
구_확정6개월_중위가격,0.38
소형60이하_비중,0.32


In [17]:
serve_month = pd.Period("2026-08", "M")
q, q_all = interval_table(serve_month, ALPHA, WINDOW)
serve_iv = q.reset_index()
serve_iv["적정가 하한 배수"] = np.exp(serve_iv["lo"]); serve_iv["적정가 상한 배수"] = np.exp(serve_iv["hi"])
serve_iv["등급 사용"] = serve_iv["단지"].eq("기존")
serve_iv.to_csv(MODEL_DIR / "interval_table.csv", index=False, encoding="utf-8-sig")

config = {
    "model": "LightGBM", "target": "log(거래금액)", "params": LGB_PARAMS,
    "features": FEATURES, "categorical": CATEGORICAL, "train_period": ["2021-07-01", "2026-06-30"],
    "service_input": {"거래유형": "중개거래", "예측 기준일": "서비스 월 1일", "확정 거래": "m-2월까지"},
    "interval": {"method": "OOF 잔차 분위수, 이동 창", "window_months": WINDOW, "alpha": ALPHA,
                 "segments": SEG, "price_band_억": [6, 9, 15], "fallback": "창 전체 분위수",
                 "table": "interval_table.csv", "table_month": str(serve_month)},
    "grade": {"rule": "구간 밖 초과분이 해당 방향 반폭 이하 → 다소, 초과 → 고/저평가", "new_complex": "등급 없음(참고용)"},
    "retrain": f"매달 (m-2월까지 확정 거래), 반복 {final_params['n_estimators']}회 고정",
}
json.dump(config, open(MODEL_DIR / "model_config.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)

metrics = {
    "rolling_monthly": {k: perf.loc[k, ["n", "MAPE", "편향"]].round(3).to_dict("index") for k in ["Valid", "Test"]},
    "rolling_overall": overall.round(3).to_dict("index"),
    "fixed_split_before_last_trade": {"Valid_MAPE": 6.160, "Test_MAPE": 9.637, "Test_MAPE_TrainValid_refit": 8.052},
    "vs_last_trade_baseline": compare.to_dict("records"),
    "naive_rule": {"Valid_MAPE": round(naive["Valid_MAPE"], 3), "Test_MAPE": round(naive["Test_MAPE"], 3)},
    "same_protocol_v1_vs_v2_test": {"V1": 13.26, "V2": 9.92},
    "interval_test": {"coverage_existing_pct": round(test_iv.loc[mk & test_iv["단지"].eq("기존"), "포함"].mean() * 100, 2),
                      "width_existing_pct": round(test_iv.loc[mk & test_iv["단지"].eq("기존"), "폭%"].mean(), 2),
                      "coverage_new_pct": round(test_iv.loc[mk & test_iv["단지"].eq("신규"), "포함"].mean() * 100, 2)},
}
json.dump(metrics, open(MODEL_DIR / "metrics.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=float)
test_iv.merge(trade[["transaction_id", "단지일련번호", "시군구명", "거래금액"]], on="transaction_id").assign(
    예측가_만원=lambda d: np.exp(d["p"]).round(0)).drop(columns=["y"]).to_parquet(MODEL_DIR / "test_predictions.parquet", index=False)
display(serve_iv.round(3))
print(sorted(p.name for p in MODEL_DIR.iterdir()))

,단지,가격대,lo,hi,적정가 하한 배수,적정가 상한 배수,등급 사용
0,기존,15억 이상,-0.081,0.086,0.922,1.090,True
1,기존,6~9억,-0.051,0.114,0.950,1.120,True
2,기존,6억 미만,-0.075,0.110,0.927,1.116,True
3,기존,9~15억,-0.047,0.115,0.954,1.122,True
4,신규,15억 이상,-0.128,0.619,0.880,1.856,False
5,신규,6~9억,-0.275,0.190,0.759,1.209,False
6,신규,6억 미만,-0.120,0.254,0.887,1.290,False
7,신규,9~15억,-0.355,0.283,0.701,1.327,False


['interval_table.csv', 'lgbm_regression_v2.txt.gz', 'metrics.json', 'model_config.json', 'oof_gap_gu_month.parquet', 'oof_predictions.parquet', 'test_predictions.parquet']


## 11. 정리

| 항목 | 결정 |
|---|---|
| 모델 | LightGBM 단일 (Optuna, 시간 fold 2개, 잎 72 · 반복 4,290), 정답 `log(거래금액)`, 모델 파일 11MB (gzip) |
| Feature | 레지스트리 기본 199개. 예상가격은 단지 자체 이력일 때만 (`예상가격_단지만`), 같은 단지 · 면적 직전 거래 5개 (`06_last_trade`) |
| 앙상블 · 정답 변환 · 최근 오차 보정 | 채택 기준 미달 → 미사용 |
| 평가 | 매달 재학습 OOF: Valid 5.33% · Test 5.82% (중개 4.95 · 5.53%). 직전 거래 있는 Test 중개: 모델 5.37% vs 직전 거래가 6.67%, 단순 규칙 14.94% |
| 오차 범위 | 최근 3개 확정월 OOF 잔차, 단지 × 가격대별 8~92% 분위수 → 기존 단지 Test 포함률 81.9%, 평균 폭 16.6% |
| 등급 | 범위 기준 5단계 규칙, 신규 단지는 등급 없음 |
| 31 입력 | `oof_gap_gu_month.parquet` (사용가능월 = 거래월 + 2) |

### 남은 한계와 처리 계획

| 한계 | 영향 | 처리 |
|---|---|---|
| 상승기 과소추정 (Test 편향 -2.1%) | 동대문 · 동작 · 성동 등 상승 지역 적정가가 3%가량 낮게 나옴 | 확정 거래 2개월 지연은 구조적 한계. 웹에 "최근 2개월 거래 미반영" 안내, 31 상승 확률과 함께 표시 |
| 신규 단지 구간 | 80% 미달 | 등급 없이 참고용 범위 (설계서 6-1 갱신) |
| 직거래 | 오차 2배, 저평가 판정 많음 | 서비스 입력은 중개거래 기준, 거래 목록에서 직거래 구분 표시 |
| 급락장 · 거래 절벽 | 2022 하반기 OOF MAPE 12~16% — 직전 거래가 5개월 전이고 그사이 9~13% 하락 | 직전 거래 경과일을 화면에 표시, 31 하락 신호와 함께 해석 |
| 2022 OOF 품질 | 초기 학습 데이터 짧음 (중개 MAPE 8.8%) | 31에서 Gap Feature Ablation |
| CatBoost 재현 불안정 | 후보 제외 | V3에서 필요 시 재검토 |
| 응급 · 유치원 거리 (현재 목록) | ablation에서 효과 없음 | 기본 Feature 제외 유지 |